# Softmax Regresyon Çalışma ve Uygulama Notları (Çok Sınıflı Lojistik Regresyon)

Bu çalışmada, çok sınıflı (multiclass) sınıflandırma problemleri için lojistik regresyon modelinin genelleştirilmiş hali olan Softmax Regresyon modelinin teorik altyapısı incelenmiş; sıfırdan NumPy, Scikit-Learn ve PyTorch uygulamaları gerçekleştirilmiştir. Çalışma adımları ve elde edilen sonuçlar aşağıda sunulmuştur.

## 1. Teorik Altyapı ve Matematiksel Notlar

Softmax regresyonda doğrusal model çıktıları, tüm sınıfların toplam olasılığını $1.0$ yapacak şekilde normalleştiren Softmax aktivasyon fonksiyonundan geçirilerek olasılık değerleri elde edilmektedir.

### 1.1 Temel Olasılık Modeli (Log-Oranlar)
Çok sınıflı sınıflandırma problemlerinde $K$ adet sınıf ($K \ge 3$) ve her bir gözlem için $p$ öznitelikli bir $\mathbf{x}_i$ vektörü bulunduğu varsayılmaktadır. Sınıflardan biri (örneğin $K$. sınıf) referans sınıfı (reference category) olarak seçilmektedir. Diğer sınıfların olasılıkları, referans sınıfının olasılığına bölünerek log-oranları (log-odds) doğrusal olarak modellenir:

$$
\log\left(\frac{P(y_i = k | \mathbf{x}_i)}{P(y_i = K | \mathbf{x}_i)}\right) = \beta_{k0} + \beta_{k1}x_{i1} + \cdots + \beta_{kp}x_{ip} = \boldsymbol{\beta}_k^T \mathbf{x}_i
$$

Bu formül, herhangi bir $k$ sınıfının referans sınıfına kıyasla seçilme olasılığının logaritmik oranını doğrusal bir fonksiyon olarak ifade etmektedir. Formüldeki bileşenler aşağıda açıklanmıştır:

*   $P(y_i = k | \mathbf{x}_i)$: $\mathbf{x}_i$ öznitelikleri bilindiğinde, $i$. gözlemin $k$ sınıfına ait olma koşullu olasılığıdır.  
    *Örnek:* Bir müşterinin orta risk grubuna (Medium = 1) ait olma olasılığı $P(y_i = 1 | \mathbf{x}_i) = 0.245$'dir.
*   $P(y_i = K | \mathbf{x}_i)$: Referans sınıfı olarak seçilen $K$. sınıfa ait olma koşullu olasılığıdır.  
    *Örnek:* Referans sınıfı düşük risk (Low = 0) olarak seçildiğinde, müşterinin düşük risk grubuna ait olma olasılığı $P(y_i = 0 | \mathbf{x}_i) = 0.055$'dir.
*   $\frac{P(y_i = k | \mathbf{x}_i)}{P(y_i = K | \mathbf{x}_i)}$: $k$ sınıfının olasılığının referans sınıfı olasılığına oranıdır (odds).  
    *Örnek:* Orta risk olasılığının düşük risk olasılığına oranı $\frac{0.245}{0.055} \approx 4.482$'dir (Orta risk olma şansı, düşük risk olma şansının yaklaşık 4.5 katıdır).
*   $\log(\cdot)$: Olasılık oranının doğal logaritmasıdır (logit dönüşümü).  
    *Örnek:* Yukarıdaki oranın doğal logaritması $\log(4.482) \approx 1.5$ olarak bulunur.
*   $\boldsymbol{\beta}_k$: $k$ sınıfı için eğitilen $(p+1) \times 1$ boyutundaki katsayılar (ağırlıklar) vektörüdür. $\beta_{k0}$ sabit terim (bias/intercept) iken, diğer katsayılar ilgili özniteliklerin ağırlıklarıdır.  
    *Örnek:* Orta risk sınıfı katsayıları $\boldsymbol{\beta}_{medium} = [0.5, 0.1, -0.05]^T$ ise; modelin sabit terimi $0.5$, yaş katsayısı $0.1$ ve gelir katsayısı $-0.05$'dir.
*   $\mathbf{x}_i$: $i$. gözleme ait öznitelikleri içeren $(p+1) \times 1$ boyutundaki tasarım vektörüdür. Sabit terim için ilk elemanı $1$ olarak atanır.  
    *Örnek:* Yaşı 25 ve geliri 30 olan bir kişi için tasarım vektörü $\mathbf{x}_i = [1, 25, 30]^T$ şeklinde oluşturulur.
*   $\boldsymbol{\beta}_k^T \mathbf{x}_i$: Katsayı vektörü ile öznitelik vektörünün nokta (dot) çarpımıdır. Doğrusal tahminci (linear predictor/logit) olarak adlandırılır.  
    *Örnek:* $\eta_{medium} = \boldsymbol{\beta}_{medium}^T \mathbf{x}_i = 0.5(1) + 0.1(25) + (-0.05)(30) = 1.5$ olarak hesaplanır.

---

### 1.2 Log-Oranlardan Olasılıklara Geçiş
Logaritma fonksiyonundan kurtulmak amacıyla eşitliğin her iki tarafının üstel (exponential) değeri alınır:

$$
\frac{P(y_i = k | \mathbf{x}_i)}{P(y_i = K | \mathbf{x}_i)} = e^{\boldsymbol{\beta}_k^T \mathbf{x}_i} \implies P(y_i = k | \mathbf{x}_i) = P(y_i = K | \mathbf{x}_i) e^{\boldsymbol{\beta}_k^T \mathbf{x}_i}
$$

Bu dönüşüm, doğrusal skorları doğrudan sınıfların birbirine göre olasılık katlarına çevirmektedir. Üstel fonksiyon ($e^z$) eksi sonsuzdan artı sonsuza uzanan ham skorları pozitif değerlere taşır; örneğin $\eta_{medium} = 1.5$ için $e^{1.5} \approx 4.482$ elde edilir.

---

### 1.3 Normalizasyon Kısıtı ve Referans Olasılığı
Tüm sınıfların olasılıklar toplamının $1.0$ olması gerekliliğinden faydalanılarak referans sınıfın olasılığı yalnız bırakılır:

$$
\sum_{j=1}^K P(y_i = j | \mathbf{x}_i) = 1 \implies P(y_i = K | \mathbf{x}_i) \left( 1 + \sum_{j=1}^{K-1} e^{\boldsymbol{\beta}_j^T \mathbf{x}_i} \right) = 1 \implies P(y_i = K | \mathbf{x}_i) = \frac{1}{1 + \sum_{j=1}^{K-1} e^{\boldsymbol{\beta}_j^T \mathbf{x}_i}}
$$

Bu eşitlik, olasılıkların toplamının $1.0$ olmasını garanti altına alarak referans sınıfının kesin olasılık değerini belirlemektedir. Paydadaki bileşenler:

*   $\sum_{j=1}^{K-1} e^{\boldsymbol{\beta}_j^T \mathbf{x}_i}$: Referans sınıfı haricindeki diğer tüm sınıfların üstel doğrusal skorlarının toplamıdır.  
    *Örnek:* 3 sınıflı sistemde referans dışındaki sınıfların (orta ve yüksek) üstelleri $e^{1.5} \approx 4.482$ ve $e^{2.55} \approx 12.807$ ise toplamları $4.482 + 12.807 = 17.289$ olur.
*   $1$: Referans sınıfın kendi üstel değerini ($e^{\eta_{low}} = e^0 = 1$) temsil eder.
*   Payda ($1 + 17.289 = 18.289$): Olasılıkların paydası olarak kullanılır ve tüm olasılıkların normalleştirilmesini sağlar.

---

### 1.4 Final Olasılık Formülü (Softmax Fonksiyonu)
Referans olasılığı yerine yazıldığında, herhangi bir $k$ sınıfına ait nihai olasılık formülü elde edilir:

$$
P(y_i = k | \mathbf{x}_i) = \frac{e^{\boldsymbol{\beta}_k^T \mathbf{x}_i}}{1 + \sum_{j=1}^{K-1} e^{\boldsymbol{\beta}_j^T \mathbf{x}_i}}
$$

Eğer referans sınıfının katsayı vektörü de sıfır ($\boldsymbol{\beta}_K = \mathbf{0}$) olarak modele dahil edilirse, tüm sınıfları kapsayan simetrik **Softmax** formülü elde edilir:

$$
P(y_i = k | \mathbf{x}_i) = \sigma(\mathbf{z}_i)_k = \frac{e^{\boldsymbol{\beta}_k^T \mathbf{x}_i}}{\sum_{j=1}^K e^{\boldsymbol{\beta}_j^T \mathbf{x}_i}}
$$

Bu formül, giriş skorlarını (logits) $0$ ile $1$ arasında değişen ve toplamları $1.0$ olan geçerli bir olasılık dağılımına dönüştürmektedir. Formüldeki pay ve payda şu şekilde yorumlanmaktadır:

*   Pay ($e^{\boldsymbol{\beta}_k^T \mathbf{x}_i}$): Hedeflenen $k$ sınıfının üstel doğrusal tahmincisidir (normalleştirilmemiş ham olasılık payı).  
    *Örnek:* Yüksek risk sınıfı için pay değeri $e^{2.55} \approx 12.807$'dir.
*   Payda ($\sum_{j=1}^K e^{\boldsymbol{\beta}_j^T \mathbf{x}_i}$): Tüm sınıfların üstel doğrusal tahmincilerinin toplamıdır (normalleştirme sabiti).  
    *Örnek:* Yukarıdaki 3 sınıf için payda $1.000 + 4.482 + 12.807 = 18.289$'dur. Buradan Yüksek Risk olasılığı $P(High) = \frac{12.807}{18.289} \approx 0.700$ olarak bulunur.

### Softmax Olasılıklarının Davranışı
Aşağıdaki ilk grafikte, bir sınıfın doğrusal tahmincisi ($\eta_1$) değişirken diğer sabit sınıfların olasılıklarının nasıl etkilendiği gösterilmiştir. İkinci grafikte ise iki doğrusal tahmincinin ($\eta_1$ ve $\eta_2$) oluşturduğu 2 boyutlu uzayda olasılıkların dağılımı görselleştirilmiştir:

<div style="display: flex; justify-content: center; align-items: center; gap: 10px; margin-top: 15px; margin-bottom: 15px;">
  <img src="figures/softmax_probabilities_vs_eta1.png" alt="Softmax Olasılıklarının Değişimi" width="360" />
  <img src="figures/softmax_2d_distribution.png" alt="2 Boyutlu Softmax Dağılımı" width="360" />
</div>

---

### 1.5 Kategorik Çapraz Entropi (Categorical Cross-Entropy) Kayıp Fonksiyonu
Model katsayılarının eğitim verisine uyumunu ölçmek amacıyla kullanılan hata kriteridir:

$$
J(W) = -\frac{1}{n} \sum_{i=1}^n \sum_{k=1}^K y_{ik} \log(P_{ik})
$$

Modelin tahmin ettiği olasılık dağılımı ile gerçek etiketler arasındaki farkı ölçerek, optimize edilecek tek bir skaler kayıp değeri elde edilmektedir. Formüldeki terimler:

*   $y_{ik}$: Gerçek sınıf göstergesidir. Eğer $i$. gözlemin gerçek sınıfı $k$ ise $1$, aksi takdirde $0$ değerini alır (one-hot encoding).  
    *Örnek:* Bir gözlemin gerçek sınıfı Yüksek Risk (High = 2) ise, bu gözlem için gösterge vektörü $\mathbf{y}_i = [y_{i, low}, y_{i, medium}, y_{i, high}] = [0, 0, 1]$ olur.
*   $P_{ik}$: Model tarafından $i$. gözlemin $k$ sınıfına ait olma olasılığına verilen tahmindir ($P(y_i = k | \mathbf{x}_i)$).  
    *Örnek:* Model, yukarıdaki kişi için $P_{i} = [P_{i, low}, P_{i, medium}, P_{i, high}] = [0.055, 0.245, 0.700]$ olasılıklarını tahmin etmiş olsun.
*   $-\sum_{k=1}^K y_{ik} \log(P_{ik})$: Tek bir gözlem için hesaplanan çapraz entropi kaybıdır (ceza değeri).  
    *Örnek:* Bu müşteri için hesaplanan kayıp:  
    $- [0 \times \log(0.055) + 0 \times \log(0.245) + 1 \times \log(0.700)] = - \log(0.700) \approx 0.356$ olarak bulunur.  
    *Eğer model yanlış tahmin yapıp bu kişiye %1 ($0.01$) olasılık verseydi,* ceza değeri $-\log(0.01) \approx 4.605$ gibi çok yüksek bir değer olacaktı. Doğru sınıfa yüksek olasılık verilmesi kaybı azaltır.

---

### 1.6 Matris Formunda Gradyan Güncellemesi
Gradyan inişi algoritması ile ağırlıkları güncellemek için kayıp fonksiyonunun türevi matris formunda hesaplanır:

$$
\nabla_W J(W) = \frac{1}{n} (P - Y)^T X
$$

Bu ifade, tüm katsayıların hatayı azaltmak için hangi yönde ve ne kadar değiştirilmesi gerektiğini matris çarpımlarıyla verimli şekilde hesaplamaktadır. Gradyan bileşenleri:

*   $P - Y$: Tahmin edilen olasılıklar ile gerçek sınıflar arasındaki farkı veren $n \times K$ boyutundaki hata matrisidir.  
    *Örnek:* Gerçek sınıfı High ($[0, 0, 1]$) olan bir kişiye model $[0.05, 0.25, 0.70]$ olasılıklarını vermişse, bu kişi için hata satırı:  
    $[0.05, 0.25, 0.70] - [0, 0, 1] = [0.05, 0.25, -0.30]$ olarak hesaplanır.  
    Modelin High sınıfına verdiği olasılık eksik kalmıştır ($-0.30$ fark); Low ve Medium sınıflarına ise gereksiz olasılık verilmiştir ($+0.05$ ve $+0.25$). Gradyan bu hataları düzeltmek için katsayıları güncelleyecektir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# görselleştirme ayarlarının yapılması
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

## 2. Elle Hesaplama Çalışması (Sayısal Örnek)

Algoritmanın işleyişini incelemek amacıyla, yaş (Age) ve gelir (Income) özelliklerine dayalı risk seviyelerini (Düşük, Orta, Yüksek) sınıflandıran 12 örnekten oluşan basit bir veri seti ele alınmıştır.

### Örnek Veri Seti
- Sınıflar: Düşük Risk (Low = 0, Referans Sınıfı), Orta Risk (Medium = 1), Yüksek Risk (High = 2).

### Verilen Katsayılar (Örnek Parametreler)
- Referans sınıfı Low (0) olarak belirlendiğinden, bu sınıfa ait katsayı vektörü sıfır olarak kabul edilmektedir: $\boldsymbol{\beta}_{low} = [0, 0, 0]^T$
- Orta Risk (1) sınıfı katsayıları: $\boldsymbol{\beta}_{medium} = [0.5, 0.1, -0.05]^T$ (sabit terim, yaş ve gelir katsayıları)
- Yüksek Risk (2) sınıfı katsayıları: $\boldsymbol{\beta}_{high} = [1.2, 0.15, -0.08]^T$ (sabit terim, yaş ve gelir katsayıları)

### Yeni Gözlem Noktası
- Yaş ($x_1$) = 25
- Gelir ($x_2$) = 30

### Adım 1: Doğrusal Tahmincilerin (Linear Predictors) Hesaplanması
$$
\eta_{medium} = 0.5 + 0.1(25) + (-0.05)(30) = 0.5 + 2.5 - 1.5 = 1.5
$$
$$
\eta_{high} = 1.2 + 0.15(25) + (-0.08)(30) = 1.2 + 3.75 - 2.4 = 2.55
$$
$$
\eta_{low} = 0.0 \quad (\text{Referans Sınıfı Kısıtı})
$$

### Adım 2: Üstel (Exponential) Değerlerin Hesaplanması
$$
e^{\eta_{medium}} = e^{1.5} \approx 4.482
$$
$$
e^{\eta_{high}} = e^{2.55} \approx 12.807
$$
$$
e^{\eta_{low}} = e^{0} = 1.000
$$

### Adım 3: Paydanın Hesaplanması (Toplam Üstel Değer)
$$
\sum_{j=1}^3 e^{\eta_j} = 1.000 + 4.482 + 12.807 = 18.289
$$

### Adım 4: Sınıf Olasılıklarının Hesaplanması (Softmax)
$$
P(Low) = \frac{1.000}{18.289} \approx 0.055 \quad (\%5.5)
$$
$$
P(Medium) = \frac{4.482}{18.289} \approx 0.245 \quad (\%24.5)
$$
$$
P(High) = \frac{12.807}{18.289} \approx 0.700 \quad (\%70.0)
$$

### Adım 5: Doğrulama ve Sınıf Tahmini
$$
P(Low) + P(Medium) + P(High) = 0.055 + 0.245 + 0.700 = 1.000 \quad (\%100) \quad \checkmark
$$
En yüksek olasılık değeri Yüksek Risk sınıfına ($P(High) = \%70.0$) ait olduğundan, gözlemin sınıfı **Yüksek Risk** olarak tahmin edilmektedir.

In [ ]:
def softmax(z):
    shift_z = z - np.max(z, axis=1, keepdims=True)
    exps = np.exp(shift_z)
    return exps / np.sum(exps, axis=1, keepdims=True)

class ScratchSoftmaxRegression:
    def __init__(self, lr=0.01, epochs=5000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.losses = []
        self.weight_history = []
        
    def _one_hot(self, y, K):
        m = len(y)
        one_hot = np.zeros((m, K))
        one_hot[np.arange(m), y] = 1
        return one_hot
        
    def fit(self, X, y):
        m, n = X.shape
        K = len(np.unique(y))
        X_b = np.c_[np.ones((m, 1)), X]
        self.weights = np.zeros((K, n + 1))
        Y_one_hot = self._one_hot(y, K)
        
        for epoch in range(self.epochs):
            logits = X_b.dot(self.weights.T)
            P = softmax(logits)
            P = np.clip(P, 1e-15, 1 - 1e-15)
            
            loss = -1/m * np.sum(Y_one_hot * np.log(P))
            self.losses.append(loss)
            self.weight_history.append(self.weights.copy())
            
            gradient = 1/m * (P - Y_one_hot).T.dot(X_b)
            self.weights -= self.lr * gradient
            
    def predict_proba(self, X):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return softmax(X_b.dot(self.weights.T))
        
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

### Sıfırdan NumPy ile Modelleme ve Görselleştirme

Yukarıda teorik gösterimi ve sayısal hesaplamaları sunulan 12 örnekli müşteri risk sınıflandırma veri seti (Düşük, Orta, Yüksek), sıfırdan NumPy kullanılarak geliştirilen `ScratchSoftmaxRegression` sınıfı ile eğitilmiştir. Eğitim süreci boyunca hesaplanan cross-entropy kaybının iterasyona göre değişimi grafikle gösterilmiştir.

In [ ]:
# 12 örnekli risk veri seti
# Age, Income, Risk Level (0: Low, 1: Medium, 2: High)
X_risk = np.array([
    [25, 30],
    [30, 40],
    [35, 50],
    [40, 60],
    [45, 70],
    [50, 80],
    [25, 35],
    [30, 45],
    [35, 55],
    [40, 65],
    [45, 75],
    [50, 85]
])
y_risk = np.array([0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2])

# Modelin doğru çalışması ve hızlı yakınsaması için ölçeklendirme yapılır
X_risk_scaled = (X_risk - np.mean(X_risk, axis=0)) / np.std(X_risk, axis=0)

model_scratch = ScratchSoftmaxRegression(lr=0.1, epochs=10000)
model_scratch.fit(X_risk_scaled, y_risk)

print("Eğitilen Katsayı Matrisi (W):\n", model_scratch.weights)

# Kayıp grafiğinin çizilmesi
plt.figure(figsize=(8, 5))
plt.plot(model_scratch.losses, 'g-', linewidth=2)
plt.title('Softmax Regresyon Kayıp (Cross Entropy) Değerinin İterasyonla Değişimi', fontsize=12)
plt.xlabel('İterasyon (Epoch)')
plt.ylabel('Kayıp (Loss)')
plt.show()

## 3. Scikit-Learn Kütüphanesi ile Modelleme

Modeli daha büyük bir veri kümesinde test etmek amacıyla 3 sınıftan oluşan 1000 örnekli sentetik bir veri seti üretilmiştir. Özellik standartlaştırma ve model eğitimi bir `Pipeline` ile birleştirilmiştir. Model başarısı `classification_report`, karmaşıklık matrisi ve çok sınıflı log-loss metrikleri ile analiz edilmiştir.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, log_loss

# 1000 örnekli, 3 sınıflı sentetik veri seti oluşturulması
X_syn, y_syn = make_classification(
    n_samples=1000, n_features=2, n_redundant=0, n_classes=3, 
    n_clusters_per_class=1, class_sep=1.5, random_state=42
)

# Veriyi eğitim ve test olarak bölme
X_train, X_test, y_train, y_test = train_test_split(
    X_syn, y_syn, test_size=0.2, random_state=42, stratify=y_syn
)

# Pipeline kurulumu (Standartlaştırma + Softmax Regresyon)
# Scikit-Learn kütüphanelerindeki LogisticRegression sınıfı çoklu sınıf durumlarında varsayılan olarak multinomial (softmax) yaklaşımını kullanmaktadır.
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='lbfgs', C=1.0, random_state=42)
)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)

print("Sınıflandırma Raporu:\n", classification_report(y_test, y_pred))
print(f"Çok Sınıflı Log-Loss: {log_loss(y_test, y_pred_proba):.4f}")

### Model Katsayıları ve Karar Sınırları Grafiklerinin Çizilmesi

Eğitilen modelin her bir sınıf için belirlediği katsayıların önem bar grafiği ve 2D karar sınırları görselleştirilmiştir.

In [ ]:
scaler = pipeline.named_steps["standardscaler"]
model_lr = pipeline.named_steps["logisticregression"]

# Katsayıların bar grafiğiyle gösterilmesi
# model_lr.coef_ matrisi (K, p) boyutundadır.
fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
classes = ['Sınıf 0', 'Sınıf 1', 'Sınıf 2']
features = ['Öznitelik 1', 'Öznitelik 2']
colors = ['#ff9999', '#66b3ff', '#99ff99']

for idx in range(3):
    bars = ax[idx].bar(features, model_lr.coef_[idx], color=colors[idx], edgecolor='black', width=0.5)
    ax[idx].axhline(0, color='black', linewidth=0.8)
    ax[idx].set_title(f'{classes[idx]} Katsayı Değerleri')
    ax[idx].set_ylabel('Katsayı Değeri')
    for bar in bars:
        yval = bar.get_height()
        ax[idx].text(bar.get_x() + bar.get_width()/2, yval + (0.05 if yval >= 0 else -0.15), f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')
plt.suptitle('Softmax Regresyon Sınıf Katsayı Önem Grafikleri', fontsize=14, y=1.05)
plt.show()

# 2D karar sınırlarının çizilmesi
plt.figure(figsize=(10, 6))
X_train_scaled = scaler.transform(X_train)

# Izgara (meshgrid) oluşturulması
x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Olasılıkların tahmin edilmesi
Z_grid = model_lr.predict(np.c_[xx.ravel(), yy.ravel()])
Z_grid = Z_grid.reshape(xx.shape)

# Karar alanlarının boyanması
plt.contourf(xx, yy, Z_grid, alpha=0.3, cmap=plt.cm.brg)
plt.scatter(X_train_scaled[y_train==0, 0], X_train_scaled[y_train==0, 1], color='red', label='Sınıf 0', alpha=0.7, edgecolors='black')
plt.scatter(X_train_scaled[y_train==1, 0], X_train_scaled[y_train==1, 1], color='green', label='Sınıf 1', alpha=0.7, edgecolors='black')
plt.scatter(X_train_scaled[y_train==2, 0], X_train_scaled[y_train==2, 1], color='blue', label='Sınıf 2', alpha=0.7, edgecolors='black')

plt.title('Standartlaştırılmış Uzayda Çok Sınıflı Karar Sınırları', fontsize=12)
plt.xlabel('Öznitelik 1 (Standartlaştırılmış)')
plt.ylabel('Öznitelik 2 (Standartlaştırılmış)')
plt.legend(fontsize=10)
plt.show()

### Karar Sınırları ve Çok Sınıflı Olasılık Yüzeyleri
Softmax regresyonunun karar alanları ve çok sınıflı olasılık yüzeylerinin 2 boyutlu uzaydaki genel gösterimi şu şekildedir (eksen etiketleri öznitelik 1 ve öznitelik 2'yi, renk yoğunlukları ise sınıf olasılıklarını temsil etmektedir):

<img src="figures/class_probability_surfaces.png" alt="Sınıf Olasılık Yüzeyleri" width="500" style="display: block; margin: auto; margin-top: 15px; margin-bottom: 15px;" />

### Doğrusal Olmayan Sınırlar İçin Polinomsal Özellikler Denemesi

Modelin doğrusal sınırlar yerine daha esnek ve doğrusal olmayan sınırları öğrenebilmesi amacıyla ikinci dereceden polinomsal öznitelik genişletmesi (`PolynomialFeatures`) eklenmiş ve performans farkı Log-Loss değeri ile karşılaştırılmıştır.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_pipeline = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    LogisticRegression(solver='lbfgs', C=1.0, random_state=42, max_iter=1000)
)
poly_pipeline.fit(X_train, y_train)

poly_pred_proba = poly_pipeline.predict_proba(X_test)
print(f"Standart Model Log Loss: {log_loss(y_test, y_pred_proba):.4f}")
print(f"Polinomsal Model Log Loss: {log_loss(y_test, poly_pred_proba):.4f}")

## 4. Uygulama Notları ve Çıkarımlar

Yukarıdaki uygulamalar sırasında dikkat edilen ve gözlemlenen bazı noktalar aşağıda özetlenmiştir:

*   **Multinomial vs. One-Vs-Rest:** Çok sınıflı problemlerde iki farklı eğitim stratejisi bulunmaktadır. Bu çalışmada kullanılan `multinomial` stratejisi tüm sınıfları tek bir Softmax olasılık dağılımı üzerinden eşzamanlı olarak eğitmektedir. Alternatif olan `ovr` (One-vs-Rest) yöntemi ise her sınıf için ayrı bir ikili lojistik model kurmaktadır. Sınıfların birbirini dışlayıcı olduğu (bir gözlem yalnızca bir sınıfa ait olabildiği) bu çalışmadaki risk sınıflandırma probleminde, multinomial yaklaşım daha tutarlı sonuçlar vermektedir.
*   **IIA (Independence of Irrelevant Alternatives) Varsayımı:** Softmax modelinde iki sınıfın olasılık oranı, üçüncü bir sınıfın varlığından bağımsızdır. "Kırmızı Otobüs / Mavi Otobüs" paradoksu olarak bilinen bu varsayım, birbirine çok benzer sınıflar olduğunda sorun yaratabilmektedir. Bu çalışmadaki risk sınıfları (Düşük, Orta, Yüksek) birbirinden yeterince farklı olduğundan bu sorun gözlemlenmemiştir.
*   **Çözücü Seçimi:** Scikit-Learn pipeline'ında `lbfgs` çözücüsü tercih edilmiştir. Bu çözücü, buradaki 1000 örnekli sentetik veri seti gibi küçük/orta ölçekli kümelerde hızlı yakınsama sağlamaktadır. Çok büyük veri setlerinde `saga` çözücüsü daha uygun olabilir.
*   **Özellik Ölçeklendirme:** Hem sıfırdan yazılan NumPy modelinde hem de Scikit-Learn pipeline'ında `StandardScaler` ile standartlaştırma uygulanmıştır. Ölçeklendirme yapılmadığında gradyan inişi çok yavaş yakınsamakta veya hiç yakınsamamaktadır; bu durum özellikle yaş ve gelir gibi farklı ölçeklerdeki özniteliklerde belirgin hale gelmektedir.

## 5. PyTorch Kütüphanesi ile Modelleme

Derin öğrenme çatısı olan PyTorch kullanılarak çok sınıflı Softmax regresyon modeli yapay sinir ağı katmanı biçiminde tasarlanmış ve eğitilmiştir. Katsayı matrisinin boyutu doğrulanmıştır.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    # Tensörlerin oluşturulması
    X_tensor = torch.tensor(X_syn, dtype=torch.float32)
    y_tensor = torch.tensor(y_syn, dtype=torch.long)  # PyTorch CrossEntropyLoss long veri tipi bekler

    class PyTorchSoftmaxRegression(nn.Module):
        def __init__(self):
            super(PyTorchSoftmaxRegression, self).__init__()
            # 2 girdi özniteliği -> 3 sınıf çıktısı (doğrusal katman)
            self.linear = nn.Linear(in_features=2, out_features=3)
            
        def forward(self, x):
            return self.linear(x)

    model_multi = PyTorchSoftmaxRegression()
    # CrossEntropyLoss fonksiyonu, girdileri logit değerleri olarak almakta ve arka planda otomatik olarak softmax ve negatif log-olabilirlik (NLL) işlemlerini uygulamaktadır.
    criterion_multi = nn.CrossEntropyLoss()
    optimizer_multi = optim.SGD(model_multi.parameters(), lr=0.1)

    epochs = 2000
    for epoch in range(epochs):
        outputs = model_multi(X_tensor)
        loss = criterion_multi(outputs, y_tensor)
        
        optimizer_multi.zero_grad()
        loss.backward()
        optimizer_multi.step()
        
        if (epoch + 1) % 500 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Kayıp: {loss.item():.4f}")
            
    # Katsayıların yazdırılması
    with torch.no_grad():
        w = model_multi.linear.weight.numpy()
        b = model_multi.linear.bias.numpy()
    print("\nPyTorch Katsayı Matrisi (w):\n", w)
    print("PyTorch Sabit Terimler (b):\n", b)
except ImportError:
    print("PyTorch kütüphanesi ortamda kurulu değil.")

## 6. Sınıflar ve Araçlar Sözlüğü (Hatırlatma Notları)

Çalışmada yararlanılan temel Scikit-Learn ve PyTorch bileşenlerinin işlevlerine dair hazırlanan hatırlatma notları:

### Scikit-Learn Bileşenleri

*   **`StandardScaler`**: Verilerin ortalamasını $0$, standart sapmasını $1$ yapacak şekilde ölçeklendirir (standartlaştırma). Özelliklerin farklı ölçeklerde olmasından kaynaklanan dengesizlikleri gidererek Softmax regresyon katsayılarının kararlı şekilde güncellenmesini ve gradyan inişinin daha hızlı yakınsamasını sağlar.
*   **`PolynomialFeatures`**: Mevcut özniteliklerin (features) birbirleriyle çarpımlarını ve yüksek dereceden üslerini alarak yeni polinomsal öznitelikler üretir. Çok sınıflı karar sınırlarının doğrusal olmayan esnek/kavisli alanlara dönüştürülebilmesine olanak sağlar.
*   **`LogisticRegression`**: Scikit-Learn'ün çok sınıflı sınıflandırma problemlerinde otomatik olarak multinomial (Softmax) yaklaşımını kullanan sınıflandırıcı modeldir. L2 (Ridge) veya L1 (Lasso) düzenlileştirme yöntemlerini ve çok sınıflı optimizasyon için uygun olan çözücüleri (`lbfgs`, `saga` vb.) destekler.
*   **`make_pipeline`**: Veri önişleme adımlarını (örneğin `StandardScaler`) ve modeli (`LogisticRegression`) tek bir ardışık düzen (pipeline) altında birleştirir. Bu sayede verinin eğitim ve test aşamalarında aynı önişleme adısından geçmesi garanti edilir ve veri sızıntısı (data leakage) engellenir.
*   **`train_test_split`**: Veri kümesini rastgele eğitim (train) ve test (test) alt kümelerine böler. `stratify=y` parametresi ile hedef sınıf oranlarının her iki kümede de korunmasını sağlar.
*   **`confusion_matrix` (Karmaşıklık Matrisi)**: Çok sınıflı sınıflandırma kararlarının sınıflar düzeyindeki dağılımını matris olarak gösterir; hangi sınıfların birbirleriyle karıştırıldığını (örneğin sınıf 1 yerine sınıf 2 tahmini) net şekilde görmeyi sağlar.
*   **`classification_report`**: Her bir sınıf için `Precision`, `Recall`, `F1-score` ve `Support` metriklerini ayrı ayrı ve ağırlıklı ortalamalarıyla birlikte özetler.
*   **`log_loss`**: Çok sınıflı sınıflandırma olasılıklarının başarısını ölçen çapraz entropi kaybı metriğidir. Gerçek sınıflara verilen olasılıkların büyüklüğünü temel alır ve modelin doğruluğunun yanında tahmin güvenilirliğini de değerlendirir.

### PyTorch Bileşenleri

*   **`nn.Linear`**: Giriş özniteliklerine doğrusal bir dönüşüm uygular ($Z = X W^T + b$). Çok sınıflı sınıflandırmada $W$ ağırlık matrisinin boyutunu (sınıf sayısı $\times$ özellik sayısı) belirler.
*   **`nn.CrossEntropyLoss`**: Çok sınıflı Çapraz Entropi (Categorical Cross Entropy) kaybını hesaplar. İçerisinde otomatik olarak bir Softmax fonksiyonu barındırdığı için sayısal olarak daha kararlıdır (`nn.NLLLoss` ve manuel Softmax kullanımına kıyasla alt/üst taşmaları önler).
*   **`optim.SGD`**: Stokastik Gradyan İnişi (Stochastic Gradient Descent) optimizasyon algoritmasıdır. Model katsayılarını hesaplanan gradyanlar ve öğrenme oranı (learning rate) doğrultusunda günceller.